# Session 1: a flame video is a stack of pictures
**A. Real-world problem.** You film a flame with a phone. How tall is it? A ruler held up to a
flickering flame is useless, but a video is just many still pictures, and a computer can
measure the same flame in every one of them.

**Plan.** (1) a picture is a grid of brightness numbers, (2) a *threshold* turns it into a
flame mask of yes/no pixels, (3) count the mask rows from the burner up, (4) turn pixels into
millimetres with a scale, (5) stack many pictures into a video and get a height for each.

Real footage is not available here, so we draw a *synthetic* flame with a height we know.
That lets us check that the measurement is right, which is the whole idea of this repo.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

ROWS, COLS = 100, 61      # picture size in pixels (rows down, columns across)
BASE_ROW = 95             # the burner top sits on this row; height is counted upward from it
AXIS_COL = 30             # the flame axis (middle column)
SOFT = 2.0                # how many pixels the flame tip fades over

## Step 1: a synthetic flame picture
Brightness runs from 0 (black) to 1 (white). Along the flame axis the brightness is 1 near
the burner and fades to 0 over a few pixels around the tip. We *define* the true height as the
place where the brightness is exactly half, so a good measurement should give that number back.

    brightness on the axis = 1 / (1 + exp((z - h) / SOFT))      z = height above the burner (px)

At z = h this is exactly 1/2. Sideways the flame is brightest in the middle and narrows upward.

In [ ]:
def flame_frame(height_px, softness=SOFT, noise=0.0, seed=0):
    """One picture (ROWS x COLS) of a flame whose half-brightness tip is height_px above the burner."""
    z = BASE_ROW - np.arange(ROWS, dtype=float)[:, None]          # height of each row above the burner
    x = np.arange(COLS, dtype=float)[None, :] - AXIS_COL           # sideways distance from the axis
    along = 1.0 / (1.0 + np.exp((z - height_px) / softness))
    half_width = np.maximum(2.0, 9.0 * (1.0 - 0.75 * z / height_px))
    img = along * np.exp(-(x / half_width) ** 2)
    img = np.where(z < 0, 0.0, img)                                # nothing below the burner row
    if noise > 0:
        img = img + np.random.default_rng(seed).normal(0.0, noise, img.shape)
    return np.clip(img, 0.0, 1.0)

## Step 2: threshold -> flame mask
A *mask* is a picture of True/False. A pixel belongs to the flame if its brightness is at least
the threshold. Then we walk up from the burner row and count how many rows still contain
flame pixels. `min_row_pixels` ignores rows with only a few stray pixels (noise specks).

In [ ]:
def flame_mask(frame, threshold):
    return frame >= threshold


def flame_height_px(mask, min_row_pixels=1):
    """Rows between the burner row and the highest row of the flame (the run must be unbroken)."""
    top = 0
    for z in range(0, BASE_ROW + 1):
        if mask[BASE_ROW - z].sum() >= min_row_pixels:
            top = z
        else:
            break
    return top

## Step 3: pixels -> millimetres
Put a ruler beside the flame. If 100 px of the picture cover 50 mm of the ruler, then
`scale = 50 / 100 = 0.5 mm per pixel`, and a flame 60 px tall is 60 x 0.5 = 30 mm tall.

In [ ]:
def scale_mm_per_px(ref_mm, ref_px):
    return ref_mm / ref_px


def height_mm(height_px, scale):
    return height_px * scale


def predicted_height_px(h_true, threshold, softness=SOFT):
    """Closed form for the noise-free measurement: the highest whole-pixel row with brightness >= threshold."""
    if threshold <= 0:
        return BASE_ROW
    if threshold >= 1:
        return 0
    z = h_true + softness * np.log((1.0 - threshold) / threshold)
    return int(min(BASE_ROW, max(0, np.floor(z + 1e-9))))

## Step 4: a video is a stack of pictures
A video of N frames is an array of shape (N, rows, columns). Measuring every frame gives a
*height trace*: height against frame number. Dividing the frame number by the frame rate
(frames per second) gives time.

In [ ]:
def video_stack(heights_px, softness=SOFT, noise=0.0, seed=0):
    return np.stack([flame_frame(h, softness, noise, seed + k) for k, h in enumerate(heights_px)])


def height_trace(stack, threshold, min_row_pixels=1):
    return np.array([flame_height_px(flame_mask(f, threshold), min_row_pixels) for f in stack])

## Step 5: see it
Left: the picture, the mask at threshold 0.5, and the measured height. Right: how the measured
height changes when you move the threshold (it is exact at 0.5 and drifts away from it).

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    h = 60
    img = flame_frame(h)
    m = flame_mask(img, 0.5)
    hp = flame_height_px(m)
    print(f"true height {h} px, measured {hp} px, scale 0.5 mm/px -> {height_mm(hp, 0.5):.1f} mm")

    fig, ax = plt.subplots(1, 3, figsize=(8.5, 4.2))
    ax[0].imshow(img, cmap="gray", vmin=0, vmax=1)
    ax[0].set_title("brightness picture")
    ax[1].imshow(m, cmap="gray")
    ax[1].set_title("mask (threshold 0.5)")
    noisy = flame_frame(h, noise=0.12, seed=3)
    ax[2].imshow(noisy, cmap="gray", vmin=0, vmax=1)
    ax[2].set_title("with camera noise")
    for a in ax:
        a.axhline(BASE_ROW - hp, color="tab:red", lw=1)
        a.set_xlabel("column (px)")
        a.set_ylabel("row (px, down)")
    fig.suptitle(f"red line: measured tip, {hp} px above the burner row")
    fig.tight_layout()
    fig.savefig(OUT / "frame_mask.png", dpi=110)

    thr = np.arange(0.1, 0.91, 0.05)
    meas = [predicted_height_px(h, t) for t in thr]
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(thr, meas, "o-", label="measured height")
    ax2.axhline(h, color="gray", ls=":", label="true height")
    ax2.set_xlabel("threshold (fraction of full brightness)")
    ax2.set_ylabel("measured height (px)")
    ax2.set_title("The threshold you pick moves the tip")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "threshold_vs_height.png", dpi=120)
    for t in (0.2, 0.5, 0.8):
        print(f"threshold {t:.1f} -> measured {predicted_height_px(h, t)} px")
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the height in `lesson.py` to 45 px and the threshold to 0.3. Predict the measured height with `predicted_height_px` first, then measure it from a drawn frame.
2. **Stretch.** Write `flame_width_px(mask, z)` that returns the number of flame pixels in the row z pixels above the burner, and print the width at z = 5, 20 and 40 for a 60 px flame.
3. **Explain.** Explain why the lesson counts the highest row z (60) rather than the number of rows (61) from the burner row to the tip.

## Assessment
- **A. Conceptual.** Why does a lower threshold give a taller flame? Is there a 'correct' threshold?
- **B. Numerical.** A ruler is 150 px long in the picture and 60 mm long in reality. A flame is 72 px tall at threshold 0.5. How tall is it in mm, and what is the uncertainty if the tip is uncertain by +-1 px?
- **C. Graph and simulation interpretation.** In the threshold figure, what height is measured at thresholds 0.2 and 0.8, and how many millimetres apart are they at 0.5 mm/px?
- **D. Design and optimisation challenge.** Design: for a flame with true height 60 px and softness 2 px, which thresholds measure the height within +-1 px of the truth? Pick one and justify your choice, with noise in mind.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    f = L.flame_frame(60)
    checks.append(("a picture has the expected shape and brightness range",
                   f.shape == (L.ROWS, L.COLS) and f.min() >= 0 and f.max() <= 1))
    m = L.flame_mask(f, 0.5)
    hp = L.flame_height_px(m)
    checks.append((f"worked example: a 60 px flame at threshold 0.5 measures 60 px, i.e. 30 mm at 0.5 mm/px (got {hp} px)",
                   hp == 60 and abs(L.height_mm(hp, L.scale_mm_per_px(50, 100)) - 30.0) < 1e-9))
    checks.append(("hand calculation: a 100 px ruler covering 50 mm gives 0.5 mm per pixel",
                   abs(L.scale_mm_per_px(50, 100) - 0.5) < 1e-12))
    # the threshold moves the tip exactly as the closed form says
    ok = True
    for h in (20, 40, 60, 80):
        for t in (0.2, 0.35, 0.5, 0.8):
            ok &= L.flame_height_px(L.flame_mask(L.flame_frame(h), t)) == L.predicted_height_px(h, t)
    checks.append(("measured height equals the closed-form prediction for 16 height/threshold pairs", bool(ok)))
    checks.append(("a lower threshold gives a taller flame, a higher one a shorter flame",
                   L.predicted_height_px(60, 0.2) > L.predicted_height_px(60, 0.5) > L.predicted_height_px(60, 0.8)))
    # edge cases
    checks.append(("edge: threshold 1.0 finds no flame (height 0)", L.flame_height_px(L.flame_mask(f, 1.0)) == 0))
    checks.append(("edge: an empty (black) picture has height 0", L.flame_height_px(L.flame_mask(np.zeros((L.ROWS, L.COLS)), 0.5)) == 0))
    checks.append(("edge: threshold 0 fills the whole column up to the top of the picture",
                   L.flame_height_px(L.flame_mask(f, 0.0)) == L.BASE_ROW and L.predicted_height_px(60, 0.0) == L.BASE_ROW))
    # a video is a stack
    heights = [50, 55, 60, 55, 50]
    stack = L.video_stack(heights)
    checks.append(("a 5-frame video is an array of shape (5, rows, cols)", stack.shape == (5, L.ROWS, L.COLS)))
    checks.append(("the height trace of the stack returns the true heights", list(L.height_trace(stack, 0.5)) == heights))
    # noise: a few stray pixels fool a naive count at a low threshold
    noisy = L.flame_frame(60, noise=0.12, seed=3)
    naive = L.flame_height_px(L.flame_mask(noisy, 0.2), 1)
    guarded = L.flame_height_px(L.flame_mask(noisy, 0.2), 3)
    checks.append((f"noise specks inflate the naive height ({naive} px) more than the guarded one ({guarded} px)",
                   naive > guarded >= 60))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
